In [1]:
import pandas as pd
import numpy as np
import h5py

In [2]:
def filter_h5_matrix(input_file, output_file, protein_coding_genes, gene_ids_file):
    print("Loading all genes from the gene IDs file...")
    all_genes = pd.read_csv(gene_ids_file)["Gene"].tolist()
    gene_to_index = {gene: idx for idx, gene in enumerate(all_genes)}
    
    print("Loading protein coding genes from the CSV file...")
    protein_coding_genes = pd.read_csv(protein_coding_genes)["Gene"].tolist()
    print(f"Total protein coding genes in CSV: {len(protein_coding_genes)}")
    
    gene_indices = sorted(list(set([
        gene_to_index[gene] 
        for gene in protein_coding_genes 
        if gene in gene_to_index
    ])))
    print(f"Number of unique gene indices: {len(gene_indices)}")
    
    print("Filtering the matrix for protein coding genes...")
    with h5py.File(input_file, "r") as h5:
        matrix = h5["final_matrix"]

        filtered_matrix = matrix[gene_indices, :]
        filtered_matrix = filtered_matrix[:, gene_indices]
  
        filtered_genes = [all_genes[idx] for idx in gene_indices]
        df = pd.DataFrame(
            filtered_matrix,
            index=filtered_genes,
            columns=filtered_genes
        )
        
        print("Saving the filtered matrix to CSV file...")
        df.to_csv(output_file)
        
        print(f"Original matrix shape: {matrix.shape}")
        print(f"Filtered matrix shape: {filtered_matrix.shape}")
        print(f"Filtered matrix saved to {output_file}")
        
        return filtered_matrix, filtered_genes

h5_file = "/home/user-kp/anugreha/human/matrices/final_matrix_erythrocyte_th=avg.h5"
gene_ids_file = "/home/user-kp/anugreha/human/gene_symbols/TS_gene_symbols.csv"
protein_coding_genes = "/home/user-kp/anugreha/human/gene_symbols/protein_coding_TS.csv"
output_file = "/home/user-kp/anugreha/human/TS_erythrocyte_avg/filtered_protein_coding.csv"
filtered_matrix, filtered_genes = filter_h5_matrix(h5_file, output_file, protein_coding_genes, gene_ids_file)

Loading all genes from the gene IDs file...
Loading protein coding genes from the CSV file...
Total protein coding genes in CSV: 19474
Number of unique gene indices: 18780
Filtering the matrix for protein coding genes...
Saving the filtered matrix to CSV file...
Original matrix shape: (58870, 58870)
Filtered matrix shape: (18780, 18780)
Filtered matrix saved to /home/user-kp/anugreha/human/TS_erythrocyte_avg/filtered_protein_coding.csv


In [3]:
df = pd.read_csv('/home/user-kp/anugreha/human/TS_erythrocyte_avg/filtered_protein_coding.csv')
shape = df.shape

print(f"The shape of the matrix is: {shape}")


The shape of the matrix is: (18780, 18781)


In [4]:
def extract_genes(filtered_csv, cancer_genes, output_csv):
    filtered_df = pd.read_csv(filtered_csv, index_col=0)
    cancer_gene_ids = pd.read_csv(cancer_genes)['Gene name'].str.strip().str.upper().drop_duplicates().tolist()

    print("Filtered DataFrame head:")
    print(filtered_df.head())
    print("Cancer gene IDs:")
    print(cancer_gene_ids[:10])
    print("Common genes:")
    common_genes = filtered_df.index[filtered_df.index.str.strip().str.upper().isin(cancer_gene_ids)]
    print(common_genes)
    
    extracted_df = filtered_df.loc[filtered_df.index.str.strip().str.upper().isin(cancer_gene_ids)]
    extracted_df.to_csv(output_csv)
    print(f"CSV file saved as {output_csv}")

filtered_csv = "/home/user-kp/anugreha/human/TS_erythrocyte_avg/filtered_protein_coding.csv"
cancer_genes = "/home/user-kp/anugreha/human/hpa_analysis/k562_downreg.csv"
output_csv = "/home/user-kp/anugreha/human/TS_erythrocyte_avg/downreg_counts.csv"

extract_genes(filtered_csv, cancer_genes, output_csv)

Filtered DataFrame head:
        OR4F5  OR4F29  OR4F16  SAMD11  NOC2L  KLHL17  PLEKHN1  PERM1   HES4  \
OR4F5   11004   11004   11004   11004  10967   10988    11003  11004  10989   
OR4F29  11004   11004   11004   11004  10967   10988    11003  11004  10989   
OR4F16  11004   11004   11004   11004  10967   10988    11003  11004  10989   
SAMD11  11004   11004   11004   11004  10967   10988    11003  11004  10989   
NOC2L   10967   10967   10967   10967  10967   10951    10966  10967  10953   

        ISG15  ...  MT-CO2  MT-ATP8  MT-ATP6  MT-CO3  MT-ND3  MT-ND4L  MT-ND4  \
OR4F5   10744  ...       0        0        0       0       0        0       0   
OR4F29  10744  ...       0        0        0       0       0        0       0   
OR4F16  10744  ...       0        0        0       0       0        0       0   
SAMD11  10744  ...       0        0        0       0       0        0       0   
NOC2L   10710  ...       0        0        0       0       0        0       0   

        MT-ND

In [18]:
def analysis(matrix_csv):
    try:
        df = pd.read_csv(matrix_csv, index_col=0)
        num_rows, num_cols = df.shape
        print(f"Shape of the matrix is: {num_rows} x {num_cols}")
       
        self_interactions = pd.DataFrame(False, index=df.index, columns=df.columns)
        for gene in df.index:
            if gene in df.columns:
                self_interactions.loc[gene, gene] = True
        num_self_interactions = self_interactions.sum().sum()
        print(f"\nNumber of possible self-interactions found: {num_self_interactions}")

        total_pairs = (num_rows * num_cols) - num_self_interactions
        print(f"Total number of possible interactions (excluding self-interactions): {total_pairs}")

        zero_counts = ((df == 0) & ~self_interactions).sum().sum()
        zero_percentage = (zero_counts / total_pairs) * 100
        
        print(f"Total zero interaction counts (excluding self-interactions): {zero_counts}")
        print(f"Percentage of zero counts (excluding self-interactions): {zero_percentage:.2f}%")
        
        non_zero_counts = total_pairs - zero_counts
        print(f"Total non-zero interaction counts: {non_zero_counts}")
        print(f"Percentage of non-zero counts: {100 - zero_percentage:.2f}%")
        
        zero_gene_rows = ((df == 0) & ~self_interactions).sum(axis=1).sort_values(ascending=False)
        top_10_rows = zero_gene_rows.head(10)
        print("\nRow genes with highest zero counts (excluding self-interactions):")
        print(top_10_rows)
   
        results = {
            'matrix_shape': (num_rows, num_cols),
            'self_interactions': num_self_interactions,
            'total_pairs_no_self': total_pairs,
            'zero_counts_no_self': zero_counts,
            'non_zero_counts': non_zero_counts,
            'zero_percentage_no_self': zero_percentage,
            'top_zero_genes_rows': top_10_rows
        }
        
        return results
        
    except FileNotFoundError:
        print(f"Error: The file {matrix_csv} was not found.")
        return None
    except Exception as e:
        print(f"An error occurred: {str(e)}")
        return None

if __name__ == "__main__":
    matrix_csv = "/home/user-kp/anugreha/human/TS_erythrocyte_avg/druggable_expression.csv"
    analysis_results = analysis(matrix_csv)

Shape of the matrix is: 4264 x 4530

Number of possible self-interactions found: 1032
Total number of possible interactions (excluding self-interactions): 19314888
Total zero interaction counts (excluding self-interactions): 19280533
Percentage of zero counts (excluding self-interactions): 99.82%
Total non-zero interaction counts: 34355
Percentage of non-zero counts: 0.18%

Row genes with highest zero counts (excluding self-interactions):
Downregulated Gene
OR5D18    4530
OR5L2     4530
OR5D3P    4530
OR5D13    4530
OR4S2     4530
TRIM51    4530
OR5D16    4530
ZNF99     4530
OR5W2     4530
OR8I2     4530
dtype: int64


In [6]:
interaction_matrix = pd.read_csv("/home/user-kp/anugreha/human/TS_erythrocyte_avg/downreg_counts.csv")
druggable_genes = pd.read_csv("/home/user-kp/anugreha/human/DGIdb/druggable_genes.csv")["Gene"].tolist()

gene_headers = interaction_matrix.columns[1:]
druggable_gene_columns = [gene for gene in gene_headers if gene in druggable_genes or gene in interaction_matrix.iloc[:, 0].tolist()]

output_data = {"Downregulated Gene": interaction_matrix.iloc[:, 0]}
sl_columns = []

for idx, row in interaction_matrix.iterrows():
    sl_genes = []
    for gene in druggable_gene_columns:
        if row[gene] == 0:
            sl_genes.append(gene)
    
    for i, gene in enumerate(sl_genes):
        col_name = f"SL{i+1}"
        if col_name not in output_data:
            output_data[col_name] = [None] * len(interaction_matrix)
        sl_columns.append(col_name)
        output_data[col_name][idx] = gene

for col in sl_columns:
    if col not in output_data:
        output_data[col] = [None] * len(interaction_matrix)

output_df = pd.DataFrame(output_data)
output_df.to_csv("/home/user-kp/anugreha/human/TS_erythrocyte_avg/druggable.csv", index=False)

In [7]:
#druggable genes expression
def filter_druggable_gene_pairs(matrix_csv, druggable_genes_csv, output_csv):
    df = pd.read_csv(matrix_csv, index_col=0)
    druggable_genes = set(pd.read_csv(druggable_genes_csv)['Gene'].tolist())
    druggable_genes_in_matrix = druggable_genes.intersection(df.columns)
    
    output_data = []
    
    for downregulated_gene in df.index:
        interaction_scores = []
        for druggable_gene in druggable_genes_in_matrix:
            interaction_score = df.loc[downregulated_gene, druggable_gene]
            interaction_scores.append(interaction_score)
        
        if interaction_scores:
            output_data.append([downregulated_gene] + interaction_scores)
    
    output_df = pd.DataFrame(output_data, columns=['Downregulated Gene'] + list(druggable_genes_in_matrix))
    output_df.to_csv(output_csv, index=False)
    
    print(f"Filtered gene pairs saved as {output_csv}")

matrix_csv = "/home/user-kp/anugreha/human/TS_erythrocyte_avg/downreg_counts.csv"
druggable_genes_csv = "/home/user-kp/anugreha/human/DGIdb/druggable_genes.csv"
output_csv = "/home/user-kp/anugreha/human/TS_erythrocyte_avg/druggable_expression.csv"

filter_druggable_gene_pairs(matrix_csv, druggable_genes_csv, output_csv)


Filtered gene pairs saved as /home/user-kp/anugreha/human/TS_erythrocyte_avg/druggable_expression.csv


In [8]:
def process_gene_interaction(input_file_path, output_file_path):
    df = pd.read_csv(input_file_path)
    
    gene_a = []
    gene_b = []
    
    for index, row in df.iterrows():
        downregulated_gene = row.iloc[0]
        
        for col in df.columns[1:]:
            if row[col] == 0 and downregulated_gene != col:
                gene_a.append(downregulated_gene)
                gene_b.append(col)
    
    final_df = pd.DataFrame({"Gene A": gene_a, "Gene B": gene_b})
    
    final_df.to_csv(output_file_path, index=False)
    print(f"Processed data saved to {output_file_path}")

input_file_path = "/home/user-kp/anugreha/human/TS_erythrocyte_avg/druggable_expression.csv"
output_file_path = "/home/user-kp/anugreha/human/TS_erythrocyte_avg/SL_table.csv"
process_gene_interaction(input_file_path, output_file_path)

Processed data saved to /home/user-kp/anugreha/human/TS_erythrocyte_avg/SL_table.csv


In [9]:
import networkx as nx

In [10]:
network_file = "/home/user-kp/anugreha/human/GRN/BioGRID_human_interactions.txt"
df = pd.read_csv(network_file,sep='\t',header =0)
G = nx.Graph()
G.add_edges_from(df.values)
genes_present = set(G.nodes)
print(df.head())

   Gene A  Gene B
0     BCR   HOXA9
1     ATM    TP53
2   NCOR1      AR
3  CTNNB1  CREBBP
4   BRCA1   CREB1


In [11]:
SL_pairs_file = "/home/user-kp/anugreha/human/TS_erythrocyte_avg/SL_table.csv"
sl_df = pd.read_csv(SL_pairs_file, sep = ',', header=0)
print(sl_df.head())
print(f"loaded {len(sl_df)} SL pairs")

  Gene A  Gene B
0  OR4F5   GPR34
1  OR4F5   ACTC1
2  OR4F5   CD244
3  OR4F5    CLK2
4  OR4F5  CXCL13
loaded 19280533 SL pairs


In [12]:
def compute_path(G, gene1, gene2):
    if gene1 not in genes_present or gene2 not in genes_present:
        return -2
    try:
        return nx.shortest_path_length(G,source=gene1,target=gene2)
    except nx.NetworkXNoPath:
        return -1
    
sl_df["Network_Distance"] = sl_df.apply(lambda row: compute_path(G, row["Gene A"], row["Gene B"]),axis=1)
sl_df.to_csv("/home/user-kp/anugreha/human/TS_erythrocyte_avg/SL_network_distance.csv",sep=',',index=False)
print("path calculated and saved!")

path calculated and saved!


In [13]:
valid_SL_df = sl_df[sl_df["Network_Distance"]==1]
valid_SL_df = valid_SL_df.sort_values(by="Network_Distance",ascending=True)
valid_SL_df.to_csv("/home/user-kp/anugreha/human/TS_erythrocyte_avg/SL_network_distance_filtered.csv",sep=',',index=False)
print("saved")

saved


In [14]:
count = (sl_df["Network_Distance"]==1).sum()
print(count)

1331


In [15]:
def filter_sl_pairs(network_file, predicted_sl_file, output_file):
    network_df = pd.read_csv(network_file)
    predicted_sl_df = pd.read_csv(predicted_sl_file)
    
    predicted_pairs = set()
    for _, row in predicted_sl_df.iterrows():
        gene_a, gene_b = row['Gene A'], row['Gene B']
        pair = tuple(sorted([gene_a, gene_b]))
        predicted_pairs.add(pair)
    
    removed_pairs = []
    filtered_rows = []
    
    for _, row in network_df.iterrows():
        gene_a, gene_b = row['Gene A'], row['Gene B']
        pair = tuple(sorted([gene_a, gene_b]))
        
        if pair in predicted_pairs:
            removed_pairs.append((gene_a, gene_b))
        else:
            filtered_rows.append(row)
    
    filtered_df = pd.DataFrame(filtered_rows)
    filtered_df.to_csv(output_file, index=False)
    
    print(f"Removed {len(removed_pairs)} pairs:")
    for pair in removed_pairs:
        print(f"- {pair[0]}, {pair[1]}")
    
    print(f"Filtered data saved to {output_file}")
    print(f"Original pairs: {len(network_df)}, Remaining pairs: {len(filtered_df)}")

network_file = "/home/user-kp/anugreha/human/TS_erythrocyte_avg/SL_network_distance_filtered.csv"
predicted_sl_file = "/home/user-kp/anugreha/human/SL_predictions/SL_predictions_merged.csv"
output_file = "/home/user-kp/anugreha/human/TS_erythrocyte_avg/SL_novel.csv"
filter_sl_pairs(network_file, predicted_sl_file, output_file)

Removed 549 pairs:
- USP9Y, KRAS
- SLITRK2, KRAS
- AMOT, CSK
- MID2, CSK
- HCRTR1, CSK
- HCRTR1, PIK3C2B
- PLA2G2A, KRAS
- PLA2G2E, MAPKAPK2
- PADI2, CSK
- UTS2, CSK
- HES5, PTEN
- HES5, BLM
- HSD3B2, KRAS
- SYCP1, CSK
- CHIA, CSK
- KCNA3, FABP4
- KCNA2, KCNA4
- KCNA2, KCNA5
- TDRD10, CSK
- NPR1, TXN
- NPR1, MMP14
- NPR1, HCRTR1
- LCE1B, FBXW7
- KPRP, KRAS
- LCE5A, KRAS
- FLG, CSK
- GJA5, TP53
- RGS8, CSK
- TEDDM1, CSK
- AXDND1, CSK
- PAPPA2, KRAS
- ITLN2, CSK
- ATP1A4, CSK
- ATP1A2, ATP4A
- ATP1A2, ATP1A3
- KCNJ10, TP53
- SLAMF8, KRAS
- KCNH1, KCNH4
- SYT14, CSK
- CAMK1G, MYC
- CAMK1G, CAMK2G
- CAMK1G, PIK3CA
- GOLT1A, CSK
- PTGS2, NFKB1
- C1orf21, CSK
- SHCBP1L, KRAS
- POMC, FGFR3
- POMC, BCAT2
- POMC, CHD1
- POMC, NOS2
- POMC, MAP3K10
- PLD5, CSK
- RGS7, CSK
- DISC1, CSK
- CAPN9, CAPN2
- MARK1, MARK3
- DUSP2, DUSP6
- REG3A, CSK
- DQX1, CSK
- NAT8B, FBXW7
- GTF2A1L, CSK
- KCNG3, KCNV1
- SLC8A1, CSK
- ALK, CDK9
- HOXD1, KRAS
- LRP2, LRP1B
- KCNH7, KRAS
- FAP, CSK
- LYPD6B, CSK
- LRP1B

In [16]:
'''
interactions_df = pd.read_csv('/home/user-kp/anugreha/human/k562_SL/gene_interactions.csv')
gene_pairs_df = pd.read_csv('/home/user-kp/anugreha/human/SL_predictions/synlethDB_expt.csv')

interaction_pairs = set()
for _, row in interactions_df.iterrows():
    pair = tuple(sorted([row['Gene A'], row['Gene B']]))
    interaction_pairs.add((pair, row['Interaction Type']))

interaction_dict = {pair: interaction_type for pair, interaction_type in interaction_pairs}

results = []
for _, row in gene_pairs_df.iterrows():
    pair = tuple(sorted([row['Gene A'], row['Gene B']]))
    interaction_type = interaction_dict.get(pair, 'Unknown')
    results.append([row['Gene A'], row['Gene B'], interaction_type])

output_df = pd.DataFrame(results, columns=['Gene A', 'Gene B', 'Interaction Type'])
output_df.to_csv('/home/user-kp/anugreha/human/TS_analysis/k562_comparison_expt_val.csv', index=False)

'''

"\ninteractions_df = pd.read_csv('/home/user-kp/anugreha/human/k562_SL/gene_interactions.csv')\ngene_pairs_df = pd.read_csv('/home/user-kp/anugreha/human/SL_predictions/synlethDB_expt.csv')\n\ninteraction_pairs = set()\nfor _, row in interactions_df.iterrows():\n    pair = tuple(sorted([row['Gene A'], row['Gene B']]))\n    interaction_pairs.add((pair, row['Interaction Type']))\n\ninteraction_dict = {pair: interaction_type for pair, interaction_type in interaction_pairs}\n\nresults = []\nfor _, row in gene_pairs_df.iterrows():\n    pair = tuple(sorted([row['Gene A'], row['Gene B']]))\n    interaction_type = interaction_dict.get(pair, 'Unknown')\n    results.append([row['Gene A'], row['Gene B'], interaction_type])\n\noutput_df = pd.DataFrame(results, columns=['Gene A', 'Gene B', 'Interaction Type'])\noutput_df.to_csv('/home/user-kp/anugreha/human/TS_analysis/k562_comparison_expt_val.csv', index=False)\n\n"